# 🏭 실전 반도체 공정 데이터 분석 (fab.csv)
### 590개 센서 · 1,500여 건의 진짜 공정 데이터로 불량 예측하기

> 이전 시간에 200건 / 11개 센서로 만든 **샘플 데이터**를 분석해봤다면,
> 이번엔 **실제 반도체 팹(fab)** 에서 수집된 진짜 데이터셋을 다뤄봅니다.
>
> 컬럼이 너무 많고, 결측값도 훨씬 많고, 이름조차 익명(`Sensor0` ~ `Sensor589`)이지만
> **이전에 배운 도구**로 차근차근 풀어갈 수 있습니다!

---

## 📚 학습 목차

| 단계 | 내용 | 새로 배우는 것 |
|:----:|------|----------------|
| 1️⃣ | 환경 설정 & 라이브러리 | (이전과 동일) |
| 2️⃣ | 데이터 불러오기 & 큰 테이블 다루기 | `info(verbose=False)` |
| 3️⃣ | 결측값 처리 — 컬럼 단위 결정 | 결측률 임계값으로 컬럼 제거 |
| 4️⃣ | 의미 없는 컬럼 제거 | **분산이 0인 센서** 찾기 |
| 5️⃣ | 특성 선택 (Feature Selection) | `SelectKBest`, ANOVA F-검정 |
| 6️⃣ | 핵심 변수 EDA | 작은 부분집합으로 시각화 |
| 7️⃣ | 데이터 전처리 & 분리 | (이전과 동일) |
| 8️⃣ | 머신러닝 모델 학습 | **`class_weight='balanced'`** |
| 9️⃣ | 모델 평가 & 해석 | 불량 클래스 Recall 최우선 |

---


## 1️⃣ 환경 설정 & 라이브러리 불러오기

> 이전 시간과 **거의 동일**합니다.
> 다만 **특성 선택** (`SelectKBest`, `f_classif`) 도구를 새로 가져옵니다.


In [ ]:
# ── 코랩 한글 폰트 설치 (최초 1회만 실행) ──────────────────
# !apt-get install -y fonts-nanum > /dev/null 2>&1
# !fc-cache -fv > /dev/null 2>&1

# ── 라이브러리 불러오기 ──────────────────────────────────────
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import SelectKBest, f_classif      # ⭐ NEW: 특성 선택
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, confusion_matrix,
                             roc_curve, auc, accuracy_score)
import warnings
warnings.filterwarnings('ignore')

# ── 그래프 스타일 설정 ───────────────────────────────────────
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['font.family'] = 'NanumGothic'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette("muted")

print("✅ 모든 라이브러리 로딩 완료!")


---
## 2️⃣ 데이터 불러오기 & 기본 탐색

> ### 📌 fab.csv 컬럼 구성
> | 컬럼 | 의미 | 비고 |
> |------|------|------|
> | `SensorTime` | 측정 시각 | 시계열 인덱스 |
> | `Sensor0` ~ `Sensor589` | 익명의 공정 센서 값 | **590개** |
> | `Pass_Fail` | **-1 = 정상(합격), 1 = 불량(불합격)** | ⚠️ **이전 샘플과 부호가 반대!** |
>
> 💡 **왜 익명인가요?**
> 실제 산업 현장의 데이터는 **영업 비밀**입니다. 이름을 가린 채 분석해도
> 통계만으로 어떤 센서가 중요한지 찾아낼 수 있어야 합니다.
>
> 💡 **왜 라벨이 반대인가요?**
> 이 데이터는 **SECOM** 이라는 공개 데이터셋의 형식을 따릅니다.
> 표준 관례에 따라 **불량(이상)을 1로** 정의합니다.


In [ ]:
# ── CSV 파일 불러오기 ────────────────────────────────────────
df = pd.read_csv('fab.csv')

# ── 기본 정보 출력 ───────────────────────────────────────────
print(f"📊 데이터 크기  : {df.shape[0]}행 × {df.shape[1]}열")
print(f"📋 첫 5컬럼    : {df.columns[:5].tolist()}")
print(f"📋 마지막 컬럼  : {df.columns[-3:].tolist()}")
print(f"\n🔍 Pass_Fail 분포 (SECOM 표준 - 1=불량, -1=정상):")
vc = df['Pass_Fail'].value_counts()
for k, v in vc.items():
    label = '❌ 불량' if k == 1 else '✅ 정상'
    print(f"   {label}({k:+d}): {v:>5d}건 ({v/len(df)*100:.1f}%)")


In [ ]:
# ── 처음 5행 미리보기 ───────────────────────────────────────
# 컬럼이 너무 많아 다 보이지 않으므로, 일부만 골라서 보기
print("📋 데이터 상위 5행 (앞쪽 6컬럼만):")
df.iloc[:, :6].head()


In [ ]:
# ── 데이터 타입 & 메모리 정보 ───────────────────────────────
# info() 그대로 호출하면 590줄이 출력되어 화면이 뒤덮입니다.
# verbose=False 옵션을 사용하면 요약 정보만 보여줍니다 ⭐
df.info(verbose=False)


In [ ]:
# ── 기술 통계 (앞 5개 센서만) ──────────────────────────────
# 590개 센서 모두에 describe()를 적용하면 결과가 너무 큽니다.
# 일부 센서만 골라서 분포를 확인합시다.
df[['Sensor0', 'Sensor1', 'Sensor2', 'Sensor3', 'Sensor4']].describe().round(2)


In [ ]:
# ── Pass_Fail 분포 시각화 ──────────────────────────────────
# ⚠️ 매우 불균형 — 불량은 전체의 6.6% 정도밖에 안 됩니다!
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

counts = df['Pass_Fail'].value_counts().sort_index()
colors = {-1: '#2ecc71', 1: '#e74c3c'}
labels = ['정상(-1)', '불량(+1)']
vals = [counts[-1], counts[1]]

# 왼쪽: 막대그래프
bars = axes[0].bar(labels, vals,
                   color=[colors[-1], colors[1]], width=0.5,
                   edgecolor='white', linewidth=2)
axes[0].set_title('정상 / 불량 건수', fontsize=13, fontweight='bold')
axes[0].set_ylabel('건수')
for bar, val in zip(bars, vals):
    axes[0].text(bar.get_x() + bar.get_width()/2,
                 bar.get_height() + 15,
                 f'{val}건', ha='center', fontweight='bold', fontsize=12)

# 오른쪽: 파이차트
total = sum(vals)
pie_labels = [f'정상\n{vals[0]}건 ({vals[0]/total*100:.1f}%)',
              f'불량\n{vals[1]}건 ({vals[1]/total*100:.1f}%)']
axes[1].pie(vals, labels=pie_labels,
            colors=[colors[-1], colors[1]],
            startangle=90, wedgeprops=dict(edgecolor='white', linewidth=2))
axes[1].set_title('정상/불량 비율', fontsize=13, fontweight='bold')

plt.suptitle('📊 Pass_Fail 분포 (불균형 데이터!)', fontsize=15, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()
print(f"\n💡 불량률은 {vals[1]/total*100:.1f}% — 모든 샘플을 '정상'이라고 답해도 정확도 {vals[0]/total*100:.1f}%!")
print("   → 그러므로 정확도(Accuracy) 한 가지만 봐선 절대 안 됩니다.")


---
## 3️⃣ 결측값 처리 — 더 적극적으로!

> 이전 샘플에서는 결측값이 적어서 모두 평균으로 채웠습니다.
> 이번엔 **컬럼별 결측률 분포**를 먼저 보고,
> **결측이 너무 많은 센서는 통째로 제거** 하는 전략을 추가합니다.

### 결측값 처리 흐름
1. 컬럼별 결측률 계산
2. 결측률이 임계값(예: 50%)을 넘는 컬럼은 **삭제**
3. 남은 컬럼의 결측치는 **중앙값**으로 채우기 (이상치에 강함)


In [ ]:
# ── 컬럼별 결측 개수 / 결측률 ──────────────────────────────
miss_cnt = df.isnull().sum()
miss_pct = (miss_cnt / len(df) * 100).round(2)

print(f"📋 전체 결측값 합계: {miss_cnt.sum():,}개")
print(f"📋 결측이 1개 이상 있는 컬럼 수: {(miss_cnt > 0).sum()}개")
print(f"📋 가장 결측 많은 컬럼 TOP 10:")
print(miss_pct.sort_values(ascending=False).head(10).to_string())


In [ ]:
# ── 결측률 분포 시각화 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# 왼쪽: 결측률 히스토그램 (전체 컬럼)
axes[0].hist(miss_pct, bins=40, color='#3498db',
             edgecolor='white', alpha=0.85)
axes[0].axvline(50, color='red', linestyle='--', linewidth=2,
                label='임계값 50%')
axes[0].set_xlabel('컬럼별 결측률 (%)')
axes[0].set_ylabel('해당하는 컬럼 수')
axes[0].set_title('📊 컬럼별 결측률 분포\n(빨간선 오른쪽 컬럼은 제거 대상)',
                  fontsize=12, fontweight='bold')
axes[0].legend()

# 오른쪽: 결측률 상위 20개 컬럼
top20 = miss_pct.sort_values(ascending=False).head(20)
colors_bar = ['#e74c3c' if v > 50 else '#f39c12' if v > 20 else '#3498db'
              for v in top20.values]
bars = axes[1].barh(top20.index, top20.values, color=colors_bar,
                    edgecolor='white', height=0.7)
axes[1].invert_yaxis()
axes[1].set_xlabel('결측률 (%)')
axes[1].set_title('🔍 결측 많은 컬럼 TOP 20', fontsize=12, fontweight='bold')
for bar, val in zip(bars, top20.values):
    axes[1].text(val + 0.5, bar.get_y() + bar.get_height()/2,
                 f'{val:.1f}%', va='center', fontsize=8)

plt.tight_layout()
plt.show()


In [ ]:
# ── 결측률 50% 초과 컬럼 제거 ───────────────────────────────
# 너무 많이 비어있는 센서는 어떻게 채워도 신뢰할 수 없으므로 통째로 제거합니다.
THRESHOLD = 50.0   # 결측률 임계값(%)

cols_to_drop = miss_pct[miss_pct > THRESHOLD].index.tolist()
print(f"🗑️  제거할 컬럼 ({len(cols_to_drop)}개):")
print(f"    {cols_to_drop[:10]}{'...' if len(cols_to_drop) > 10 else ''}")

df = df.drop(columns=cols_to_drop)
print(f"\n✅ 컬럼 정리 후 크기: {df.shape}")


In [ ]:
# ── 남은 결측값을 중앙값으로 채우기 ────────────────────────
# 💡 평균보다 중앙값이 이상치에 강합니다.
# ⚠️ 타겟(Pass_Fail)은 절대 채우지 않습니다!

numeric_cols = df.select_dtypes(include='number').columns.tolist()
numeric_cols.remove('Pass_Fail')

before = df.isnull().sum().sum()
df[numeric_cols] = df[numeric_cols].fillna(df[numeric_cols].median())
after = df.isnull().sum().sum()

print(f"✅ 결측값 처리 완료!")
print(f"   처리 전: {before:,}개  →  처리 후: {after}개")
print(f"   현재 데이터 크기: {df.shape}")


---
## 4️⃣ 의미 없는 컬럼 제거 — 분산이 0인 센서

> 실제 데이터에는 **항상 같은 값만 기록되는 센서**가 종종 있습니다.
> (장비가 꺼져있거나, 항상 0을 출력하거나…)
>
> 분산(variance)이 0이면 그 컬럼은 **모델 학습에 전혀 도움이 안 됩니다**.
> → 제거합시다.


In [ ]:
# ── 분산이 0인 컬럼(상수 컬럼) 찾아서 제거 ─────────────────
# 💡 var() 가 0이면 모든 값이 같다는 뜻!

variances = df[numeric_cols].var()
constant_cols = variances[variances == 0].index.tolist()
near_constant = variances[(variances > 0) & (variances < 1e-6)].index.tolist()

print(f"📊 분산 = 0 인 컬럼: {len(constant_cols)}개")
print(f"📊 분산이 매우 작은(< 1e-6) 컬럼: {len(near_constant)}개")

# 둘 다 제거
to_drop_var = constant_cols + near_constant
df = df.drop(columns=to_drop_var)

# numeric_cols 도 갱신
numeric_cols = [c for c in numeric_cols if c not in to_drop_var]

print(f"\n✅ 정리 후 데이터 크기: {df.shape}")
print(f"   사용 가능한 센서 수: {len(numeric_cols)}개")


---
## 5️⃣ 특성 선택 (Feature Selection) — 중요한 센서만 추리기

> 센서가 아직 수백 개나 됩니다. 이 모두를 시각화하기는 불가능하므로,
> **타겟(Pass_Fail)과 가장 관련이 깊은 센서**만 추려냅니다.
>
> ### `SelectKBest` + `f_classif` (ANOVA F-검정)
> - 각 센서의 **정상 그룹 평균** vs **불량 그룹 평균** 차이를 통계적으로 측정
> - F-값이 클수록 두 그룹 간 차이가 크다 = **타겟 예측에 유용한 센서**
>
> 우리는 상위 **20개**만 골라서 자세히 분석해 봅시다.


In [ ]:
# ── 타겟(0/1) 만들기: 불량(1)을 양성 클래스로 ──────────────
# 💡 이전 노트북에서는 합격을 1로 두었지만,
#    이상 탐지에서는 "찾고 싶은 사건(불량)"을 1로 두는 것이 자연스럽습니다.

y = (df['Pass_Fail'] == 1).astype(int)   # 1=불량, 0=정상
X_all = df[numeric_cols].copy()

print(f"X_all 크기: {X_all.shape}")
print(f"y 분포     : 정상(0)={int((y==0).sum())}건, 불량(1)={int((y==1).sum())}건")


In [ ]:
# ── ANOVA F-검정으로 상위 K개 센서 선택 ────────────────────
K = 20
selector = SelectKBest(score_func=f_classif, k=K)
selector.fit(X_all, y)

# 점수 저장
f_scores = pd.Series(selector.scores_, index=numeric_cols)
f_scores = f_scores.replace([np.inf, -np.inf], np.nan).dropna()
top_k_cols = f_scores.sort_values(ascending=False).head(K).index.tolist()

print(f"🏆 상위 {K}개 센서 (F-값 큰 순):")
for i, col in enumerate(top_k_cols, 1):
    print(f"  {i:2d}. {col:>10s}   F={f_scores[col]:8.2f}")


In [ ]:
# ── 상위 K개 센서 F-점수 시각화 ────────────────────────────
top = f_scores.sort_values(ascending=False).head(K)

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = ['#e74c3c' if i < 5 else '#f39c12' if i < 10 else '#3498db'
              for i in range(len(top))]
bars = ax.barh(top.index[::-1], top.values[::-1],
               color=colors_imp[::-1], edgecolor='white', height=0.7)
ax.set_xlabel('F-값 (클수록 정상 vs 불량 차이가 큼)')
ax.set_title(f'🏆 ANOVA F-검정 — 상위 {K}개 센서',
             fontsize=13, fontweight='bold')

for bar, val in zip(bars, top.values[::-1]):
    ax.text(val + max(top.values)*0.01,
            bar.get_y() + bar.get_height()/2,
            f'{val:.1f}', va='center', fontsize=9)

from matplotlib.patches import Patch
legend_el = [Patch(facecolor='#e74c3c', label='최상위 5개'),
             Patch(facecolor='#f39c12', label='6~10위'),
             Patch(facecolor='#3498db', label='11~20위')]
ax.legend(handles=legend_el, loc='lower right')
plt.tight_layout()
plt.show()


---
## 6️⃣ 핵심 변수 EDA — 상위 센서 자세히 보기

> 590개를 모두 시각화하는 건 불가능했지만,
> 위에서 추린 **상위 20개 센서**라면 한눈에 비교할 수 있습니다!


In [ ]:
# ── 상위 K개 센서 그룹별 평균 비교 ─────────────────────────
group_mean = df.groupby('Pass_Fail')[top_k_cols].mean().round(3)
group_mean.index = ['✅ 정상(-1)', '❌ 불량(+1)']

# 평균 차이 시각화 (정상 - 불량)
diff = (group_mean.loc['❌ 불량(+1)'] - group_mean.loc['✅ 정상(-1)'])
diff_pct = (diff / group_mean.loc['✅ 정상(-1)'].abs().replace(0, np.nan) * 100).fillna(0)
diff_pct_sorted = diff_pct.sort_values()

fig, ax = plt.subplots(figsize=(11, 7))
colors = ['#e74c3c' if v > 0 else '#3498db' for v in diff_pct_sorted.values]
bars = ax.barh(diff_pct_sorted.index, diff_pct_sorted.values,
               color=colors, edgecolor='white', height=0.6)
ax.axvline(0, color='black', linewidth=1.5, linestyle='--')
ax.set_xlabel('불량 평균 - 정상 평균 (정상 평균 대비 %)')
ax.set_title('상위 센서 — 정상 vs 불량 평균값 차이\n(빨강=불량 시 더 높음 / 파랑=불량 시 더 낮음)',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── 상위 6개 센서 박스플롯 비교 ────────────────────────────
top6 = top_k_cols[:6]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, col in enumerate(top6):
    sns.boxplot(x='Pass_Fail', y=col, data=df,
                hue='Pass_Fail',
                palette={-1: '#2ecc71', 1: '#e74c3c'},
                legend=False,
                ax=axes[i], width=0.5)
    axes[i].set_title(f'{col}  (F={f_scores[col]:.1f})',
                      fontsize=11, fontweight='bold')
    axes[i].set_xticklabels(['정상(-1)', '불량(+1)'])
    axes[i].set_xlabel('')

plt.suptitle('📦 박스플롯 — 상위 6개 센서의 정상 vs 불량 분포',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.show()


In [ ]:
# ── 상위 K개 센서들 사이의 상관관계 히트맵 ─────────────────
# 💡 너무 강한 상관관계(>0.95)를 보이는 두 센서는 사실상 같은 정보입니다.
corr_top = df[top_k_cols + ['Pass_Fail']].corr().round(2)

plt.figure(figsize=(13, 10))
mask = np.triu(np.ones_like(corr_top, dtype=bool))
sns.heatmap(corr_top, annot=True, fmt='.2f',
            cmap='RdYlGn', center=0,
            mask=mask,
            linewidths=0.5, linecolor='white',
            annot_kws={'size': 8})
plt.title('🔗 상위 센서 간 상관관계 + Pass_Fail 과의 상관',
          fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 7️⃣ 데이터 전처리 & 분리

> 이제 **상위 K개 센서만** 입력으로 사용해 모델을 학습합니다.
> 절차는 이전 시간과 똑같습니다:
> 1. X(특성) / y(타겟) 분리
> 2. `train_test_split` (불균형이므로 `stratify=y` 필수!)
> 3. `StandardScaler` 적용


In [ ]:
# ── 학습용 X, y 만들기 ──────────────────────────────────────
X = df[top_k_cols].copy()       # 상위 K개 센서만 사용
# y 는 위에서 이미 만들었음 (1=불량, 0=정상)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y                  # ⭐ 불균형이므로 비율 유지 필수!
)

print(f"📦 학습 데이터: {X_train.shape}, 그 중 불량 {int(y_train.sum())}건")
print(f"📦 테스트 데이터: {X_test.shape}, 그 중 불량 {int(y_test.sum())}건")


In [ ]:
# ── 스케일링 ────────────────────────────────────────────────
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

# 시각화 — 스케일링 전/후 비교 (앞 8개 컬럼만)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
X_train.iloc[:, :8].boxplot(ax=axes[0], vert=False)
axes[0].set_title('스케일링 전\n(센서마다 단위가 다름)', fontsize=12, fontweight='bold')
pd.DataFrame(X_train_scaled[:, :8],
             columns=X_train.columns[:8]).boxplot(ax=axes[1], vert=False)
axes[1].set_title('스케일링 후\n(평균 0, 표준편차 1)', fontsize=12, fontweight='bold')
axes[1].axvline(0, color='red', linestyle='--', alpha=0.5)
plt.suptitle('🔧 StandardScaler 효과', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()


---
## 8️⃣ 머신러닝 모델 학습 — 불균형 처리 추가

> 이전 시간의 두 모델(로지스틱 회귀, 랜덤 포레스트)을 그대로 씁니다.
> 단, **불량률이 6.6%** 로 매우 적기 때문에
> `class_weight='balanced'` 옵션을 켜서 **소수 클래스(불량)에 가중치**를 줍니다.
>
> 그렇지 않으면 모델은 **"전부 정상"** 이라고만 외쳐도 93%의 정확도를 얻기 때문에
> 불량을 전혀 못 잡아냅니다.


In [ ]:
# ── 두 모델 학습 ────────────────────────────────────────────
# 1) 로지스틱 회귀
lr_model = LogisticRegression(
    random_state=42, max_iter=1000,
    class_weight='balanced'   # ⭐ NEW
)
lr_model.fit(X_train_scaled, y_train)

# 2) 랜덤 포레스트
rf_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=8,
    random_state=42,
    class_weight='balanced'   # ⭐ NEW
)
rf_model.fit(X_train_scaled, y_train)

print("✅ 모델 학습 완료!")
print(f"   로지스틱 회귀: 학습 반복 {lr_model.n_iter_[0]}회")
print(f"   랜덤 포레스트: 트리 {rf_model.n_estimators}개")


---
## 9️⃣ 모델 평가 — 불량 클래스 Recall이 핵심!

> **Recall(불량) = 실제 불량 중 모델이 잡아낸 비율**
>
> 반도체 공정에서 불량을 정상이라고 판정해 출하하면(False Negative)
> 큰 손해이므로, **Recall을 최대한 높여야** 합니다.


In [ ]:
# ── 정확도 / 혼동행렬 ──────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

for ax, (name, model) in zip(axes, [('로지스틱 회귀', lr_model),
                                      ('랜덤 포레스트', rf_model)]):
    y_pred = model.predict(X_test_scaled)
    cm = confusion_matrix(y_test, y_pred)
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                ax=ax, linewidths=2, linecolor='white',
                xticklabels=['정상(0)', '불량(1)'],
                yticklabels=['정상(0)', '불량(1)'],
                annot_kws={'size': 16, 'weight': 'bold'})
    acc = accuracy_score(y_test, y_pred)
    ax.set_title(f'{name}\n정확도: {acc:.1%}', fontsize=13, fontweight='bold')
    ax.set_xlabel('예측값'); ax.set_ylabel('실제값')

plt.suptitle('🎯 혼동행렬 (Confusion Matrix)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()


In [ ]:
# ── 분류 성능 리포트 (Precision / Recall / F1) ─────────────
print("=" * 60)
print("📋 로지스틱 회귀")
print("=" * 60)
print(classification_report(y_test, lr_model.predict(X_test_scaled),
                             target_names=['정상(0)', '불량(1)']))

print("=" * 60)
print("📋 랜덤 포레스트")
print("=" * 60)
print(classification_report(y_test, rf_model.predict(X_test_scaled),
                             target_names=['정상(0)', '불량(1)']))

print("\n💡 [중요] '불량(1)' 행의 recall 값을 보세요!")
print("   class_weight='balanced' 덕분에 소수 클래스 탐지율이 크게 올랐을 겁니다.")


In [ ]:
# ── ROC 커브 ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 7))

for name, model, color in [('로지스틱 회귀', lr_model, '#3498db'),
                             ('랜덤 포레스트', rf_model, '#e74c3c')]:
    y_prob = model.predict_proba(X_test_scaled)[:, 1]
    fpr, tpr, _ = roc_curve(y_test, y_prob)
    roc_auc = auc(fpr, tpr)
    ax.plot(fpr, tpr, color=color, linewidth=2.5,
            label=f'{name} (AUC = {roc_auc:.3f})')

ax.plot([0, 1], [0, 1], 'k--', linewidth=1.5, alpha=0.5,
        label='랜덤 모델 (AUC = 0.500)')
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate (= 불량 Recall)', fontsize=12)
ax.set_title('📈 ROC 커브 비교\n(AUC가 1에 가까울수록 좋은 모델)',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=11, loc='lower right')
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


In [ ]:
# ── 랜덤 포레스트 — 특성 중요도 ────────────────────────────
imp = pd.Series(rf_model.feature_importances_,
                index=top_k_cols).sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
colors_imp = ['#e74c3c' if v >= imp.quantile(0.7)
              else '#f39c12' if v >= imp.quantile(0.4)
              else '#95a5a6' for v in imp.values]
bars = ax.barh(imp.index, imp.values, color=colors_imp,
               edgecolor='white', height=0.6)
ax.set_xlabel('Feature Importance', fontsize=12)
ax.set_title('🌲 랜덤 포레스트 — 특성 중요도 (Top 20 중에서)',
             fontsize=13, fontweight='bold')
for bar, val in zip(bars, imp.values):
    ax.text(val + max(imp.values)*0.005,
            bar.get_y() + bar.get_height()/2,
            f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.show()

print("\n📊 특성 중요도 TOP 5:")
for col, val in imp.sort_values(ascending=False).head(5).items():
    print(f"  {col}: {val:.4f}   (F-검정 점수: {f_scores[col]:.1f})")


---
## 🎓 수업 정리

### ✅ 이전 시간 + 오늘 새로 배운 것

| 단계 | 이전 (샘플) | 오늘 (실전) |
|------|-------------|-------------|
| 데이터 크기 | 200 × 11 | **1,567 × 592** |
| 결측 처리 | `fillna(mean())` | **결측률 임계값 + median** |
| 차원 축소 | (불필요) | **분산 0 컬럼 제거** |
| 특성 선택 | (불필요) | **`SelectKBest` + `f_classif`** |
| 클래스 불균형 | 90:10 | **93:7 — `class_weight='balanced'`** |
| 라벨 의미 | 1=합격 | **1=불량 (SECOM 표준)** |

### 💡 핵심 인사이트

> 1. 익명 센서라도 **통계 기법** 만으로 핵심 변수를 추려낼 수 있습니다.
> 2. 컬럼이 많을수록 **결측 처리 → 분산 0 제거 → 특성 선택**의 3단 콤보가 효과적입니다.
> 3. 매우 불균형한 데이터에서 정확도(Accuracy)는 거의 무용지물입니다.
>    → **Recall, F1, ROC-AUC, PR-AUC** 를 봐야 합니다.
> 4. `class_weight='balanced'` 한 줄로 소수 클래스 탐지율을 크게 끌어올릴 수 있습니다.

### 🚀 더 나아가기
- **PCA / UMAP** 등 비선형 차원 축소
- **SMOTE** 로 불량 샘플 오버샘플링
- **XGBoost, LightGBM** 으로 성능 개선
- **GridSearchCV** 로 하이퍼파라미터 탐색
- **SHAP value** 로 개별 예측 근거 해석
